# How to Benchmark Optimization Algorithms

Benchmarking optimization algorithms is an important step when developing a new algorithm or when searching for an algorithm that is good at solving a particular problem. 

In general, benchmarking constists of the following steps:

1. Define the test problems (or get pre-implemented ones)
2. Define the optimization algorithms and the tuning parameters you want to try
3. Run the benchmark
4. Plot the results

optimagic helps you with all of these steps!

## 1. Get Test Problems

optimagic includes the problems of [Moré and Wild (2009)](https://doi.org/10.1137/080724083) as well as [Cartis and Roberts](https://arxiv.org/abs/1710.11005).

`get_benchmark_problems` returns a dictionary that maps the names of the problems to `BenchmarkProblem` objects. Below we load a subset of the Moré and Wild problems.

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook_connected"

import optimagic as om

In [ ]:
problems = om.get_benchmark_problems("example")

Each problem consists of the `function` that is minimized, the `start_params` and, if it is known, the `solution`. Let's look at the Rosenbrock problem:

In [ ]:
problem = problems["rosenbrock_good_start"]
problem.function, problem.start_params, problem.solution

The same function can be used in several problems. For example, the full Moré and Wild set contains a second Rosenbrock problem that starts further away from the solution.

### Select problems

Because the problems are stored in a dictionary, you can select or combine problems with standard Python tools. Problems and their functions have properties for this. They correspond to the filters you can use to select algorithms:

| Property | Meaning |
|---|---|
| `problem.n_params` | Number of parameters |
| `problem.has_bounds` | At least one parameter has a finite bound |
| `problem.has_nonlinear_constraints` | The problem has nonlinear constraints |
| `problem.function.problem_type` | Scalar, least-squares or likelihood problem |
| `problem.function.differentiable` | The function is differentiable |
| `problem.function.has_jac` | A closed form derivative is available |
| `problem.function.multimodal` | The function has local minima that are not global, i.e. it is a test case for global optimizers |
| `problem.function.convex` | The function is convex |

`differentiable`, `multimodal` and `convex` are `None` if the property is not known for a function. For example, this keeps the differentiable problems with at most four parameters:

In [ ]:
small_problems = {
    name: problem
    for name, problem in problems.items()
    if problem.n_params <= 4 and problem.function.differentiable
}
list(small_problems)

### Make the problems harder

You can add noise to the function values and rescale the parameters to see how well optimizers cope with these difficulties. `noise=True` and `scaling=True` use default settings. For more control, pass a noise object or a scaler. Noise objects can be combined with `+`.

In [ ]:
hard_problems = om.get_benchmark_problems(
    "example",
    noise=om.AdditiveBenchmarkNoise(std=0.1) + om.MultiplicativeBenchmarkNoise(),
    scaling=om.LinspaceBenchmarkScaler(min_scale=0.1, max_scale=10),
    seed=0,
)

## 2. Specify the Optimizers

To select the optimizers you want to benchmark, create a dictionary that maps a name of your choice to an algorithm. The values can be algorithm classes, algorithm names or configured algorithms such as `om.algos.scipy_neldermead(stopping_maxfun=500)`. This lets you compare different options of the same algorithm.

In [ ]:
optimizers = {
    "dfols": om.algos.nag_dfols,
    "nelder_mead": om.algos.scipy_neldermead,
    "truncated_newton": om.algos.scipy_truncated_newton,
}

If you need to set more than the algorithm, e.g. options for numerical derivatives, use `om.OptimizeOptions(algorithm=..., numdiff_options=...)` as value.

## 3. Run the Benchmark

Once you have your problems and your optimizers set up, you can simply use `run_benchmark`. The results are a dictionary with one `BenchmarkResult` for each (problem, optimizer) combination. Each result contains the history of the parameters and function values and the result of the optimization.

In [ ]:
results = om.run_benchmark(
    problems,
    optimizers,
)

In [ ]:
result = results[("rosenbrock_good_start", "dfols")]
result.fun_history[:5]

## 4a. Profile plots

**Profile Plots** compare optimizers over a whole problem set. 

The literature distinguishes **data profiles** and **performance profiles**. Data profiles use a normalized runtime measure whereas performance profiles use an absolute one. The profile plot does not normalize runtime by default. To do this, simply set `normalize_runtime` to True. For background information, check [Moré and Wild (2009)](https://doi.org/10.1137/080724083). 

In [ ]:
fig = om.profile_plot(
    problems=problems,
    results=results,
)

fig.show()

:::{note}

For details on using other plotting backends, see [How to change the plotting backend](how_to_change_plotting_backend.ipynb).

:::

The x axis shows runtime per problem. The y axis shows the share of problems each algorithm solved within that runtime. Thus, higher and further to the left values are desirable. Higher means more problems were solved and further to the left means, the algorithm found the solutions earlier. 

You can choose:

- whether to use `n_evaluations` or `walltime` as **`runtime_measure`**
- whether to normalize runtime such that the runtime of each problem is shown as a multiple of the fastest algorithm on that problem
- how to determine when an evaluation is close enough to the optimum to be counted as converged. Convergence is always based on some measure of distance between the true solution and the solution found by an optimizer. Whether distiance is measured in parameter space, function space, or a combination of both can be specified. 

Below, we consider a problem to be solved if the distance between the parameters found by the optimizer and the true solution parameters are at most 0.1% of the distance between the start parameters and true solution parameters. 

In [ ]:
fig = om.profile_plot(
    problems=problems,
    results=results,
    runtime_measure="n_evaluations",
    stopping_criterion="x",
    x_precision=0.001,
)

fig.show()

## 4b. Convergence plots

**Convergence Plots** look at particular problems and show the convergence of each optimizer on each problem. 

In [ ]:
fig = om.convergence_plot(
    problems=problems,
    results=results,
    n_cols=2,
    problem_subset=["rosenbrock_good_start", "box_3d"],
)

fig.show()

The further to the left and the lower the curve of an algorithm, the better that algorithm performed.

Often we are more interested in how close each algorithm got to the true solution in parameter space, not in criterion space as above. For this. we simply set the **`distance_measure`** to `parameter_space`. 

In [ ]:
fig = om.convergence_plot(
    problems=problems,
    results=results,
    n_cols=2,
    problem_subset=["rosenbrock_good_start", "box_3d"],
    distance_measure="parameter_distance",
    stopping_criterion="x",
)

fig.show()

## 5a. Convergence report

The **Convergence Report** shows for each problem and optimizer which problems the optimizer solved successfully, failed to do so, or where it stopped with an error. The respective strings are "success", "failed", or "error".
Moreover, the last column of the ```pd.DataFrame``` displays the number of dimensions of the benchmark problem.

In [ ]:
df = om.convergence_report(
    problems=problems,
    results=results,
    stopping_criterion="y",
    x_precision=1e-4,
    y_precision=1e-4,
)

In [ ]:
df

## 5b. Rank report

The **Rank Report** shows the ranks of the algorithms for each problem; where 0 means the algorithm was the fastest on a given benchmark problem, 1 means it was the second fastest and so on. If an algorithm did not converge on a problem, the value is "failed". If an algorithm did encounter an error during optimization, the value is "error".

In [ ]:
df = om.rank_report(
    problems=problems,
    results=results,
    runtime_measure="n_evaluations",
    stopping_criterion="y",
    x_precision=1e-4,
    y_precision=1e-4,
)

In [ ]:
df

## 5b. Traceback report

The **Traceback Report** shows the tracebacks returned by the optimizers if they encountered an error during optimization. The resulting ```pd.DataFrame``` is empty if none of the optimizers terminated with an error, as in the example below.

In [ ]:
df = om.traceback_report(problems=problems, results=results)

In [ ]:
df

## 6. Benchmark your own problems

To create your own problem, define a benchmark function and combine it with start parameters in a `BenchmarkProblem`. A benchmark function is a subclass of `BenchmarkFunction` that implements `fun`. Mark the class with `om.mark.least_squares` or `om.mark.likelihood` if `fun` returns least-squares residuals or likelihood contributions. A function is not tied to one number of parameters; the number of parameters is fixed by the start parameters of the problem.

In [ ]:
import numpy as np


@om.mark.least_squares
class ShiftedSphere(om.BenchmarkFunction):
    differentiable = True
    convex = True

    def fun(self, x):
        # the residuals; optimizers minimize the sum of their squares
        return x - 1


own_problems = {
    f"shifted_sphere_{n}": om.BenchmarkProblem(
        function=ShiftedSphere(),
        start_params=np.zeros(n),
        solution=om.BenchmarkSolution(fun=0.0, params=np.ones(n), exact=True),
    )
    for n in [2, 10, 50]
}

own_results = om.run_benchmark(own_problems, optimizers)
om.convergence_report(own_problems, own_results)

The `solution` is optional. If you leave it out, plots and reports compare each optimizer to the best function value any of the optimizers found. If the solution is the result of a long optimization run and not known exactly, leave `exact` at its default `False`. It is then replaced automatically if an optimizer finds a better value.

If you have a closed form derivative, override `jac(solver_type)`. It returns the derivative function for one type of optimizer, e.g. the jacobian of the residuals for least-squares optimizers and the gradient of the sum of squared residuals for scalar optimizers, or `None` if there is no closed form derivative for that type.

To turn an existing Python function into a benchmark function without writing a class, use `om.CallableBenchmarkFunction`.